# Supabase einrichten

Die öffentliche Projekt-URL und der Publishable Key sind in
`lib/cloud/cloud_config.dart` hinterlegt. Das sind Client-Zugangsdaten, keine
Verwaltungsschlüssel. Niemals `service_role`, Secret Keys oder das
Datenbankpasswort in die Flutter-App übernehmen.

## 1. Datenbank und Bilder aktivieren

1. Im Supabase-Projekt **SQL Editor → New query** öffnen.
2. Den vollständigen Inhalt von
   `supabase/migrations/202609150001_accounts_and_sync.sql` einfügen.
3. **Run** ausführen. Das Skript legt Tabellen, private Bildablage,
   Zugriffsregeln und die Funktion für versionsgeprüfte Schreibzugriffe an.
4. In Authentication anonyme Anmeldung und E-Mail-Anmeldung aktivieren;
   E-Mail-Bestätigung eingeschaltet lassen.

Die Tabellen brauchen keine manuelle Bearbeitung. Direkte Schreibzugriffe
aus dem Client sind gesperrt. Nur die Funktion `save_bike_document` schreibt
mit der geprüften Nutzer-ID aus dem Anmeldetoken. Anonyme angemeldete Nutzer
verwenden wie registrierte Nutzer die Datenbankrolle `authenticated`.

## 2. E-Mail-Codes einrichten

Die App verwendet eingegebene Codes, damit die Verknüpfung auch ohne
App-Links auf Android, iOS, Desktop und Web funktioniert.

Unter **Authentication → Email → Templates** in den Vorlagen
**Change Email Address** und **Reset Password** den Code anzeigen:

```html
<h2>Bike Setup Tracker</h2>
<p>Dein Bestätigungscode lautet:</p>
<p><strong>{{ .Token }}</strong></p>
<p>Gib diesen Code in der App ein. Teile ihn nicht mit anderen Personen.</p>
```

Vorhandene Betreffzeilen können erhalten bleiben. Einen Bestätigungslink
benötigt dieser Ablauf nicht. In der App zunächst den Code anfordern und dann
Code und gewünschtes Passwort eingeben. Beim Verknüpfen bleibt die Nutzer-ID
unverändert. Scheitert nur das Setzen des Passworts nach bestätigter E-Mail,
kann es auf der Kontoseite erneut gesetzt werden.

Für echte Nutzer **Custom SMTP** konfigurieren. Der Supabase-Testversand ist
eingeschränkt und reicht nicht für öffentliche Registrierung. Den Dienst,
Absender und Zugangsdaten richtet der Projektinhaber im Dashboard ein;
SMTP-Zugangsdaten gehören nicht in die App.

## 3. App starten und kontrollieren

Nach Paketinstallation die App vollständig neu starten (kein bloßes Hot Reload,
da neue Plattform-Plugins hinzugekommen sind).

1. Einstellungen → **Konto & Datensicherung** öffnen.
2. **Jetzt synchronisieren** wählen; der Status soll auf **Mit Cloud
   synchronisiert** wechseln. Bei fehlendem SQL bleibt die App lokal benutzbar.
3. Ein Bike, ein Setup und ein eigenes Bild anlegen; erneut synchronisieren.
4. E-Mail verknüpfen, Code bestätigen und Passwort speichern.
5. Auf einem zweiten Gerät oder in einem getrennten Browserprofil
   **Ich habe bereits ein Konto** wählen und anmelden. Gastimport deaktiviert
   lassen, wenn dort nur die Demo-Bikes vorhanden sind.
6. Bike, Setup, eigene Felder, Historie und Bild prüfen.
7. Offline ändern, Verbindung wiederherstellen und die Übertragung prüfen.
8. Dasselbe Bike auf zwei Geräten offline unterschiedlich bearbeiten: Bei einem
   Konflikt **Beide Fassungen behalten** testen.
9. Datei exportieren und auf einem anderen Gerät als Kopien importieren.

Diese Prüfung benötigt eingerichtete Tabellen und funktionierenden E-Mail-Versand.
Lokale automatisierte Tests ersetzen den Test gegen das echte Projekt nicht.

## Datenmodell und Grenzen dieser Version

- Lokal: transaktionaler Sembast-Speicher, auf Web IndexedDB, sonst im
  Dokumentenverzeichnis. Nutzerdaten und Synchronisierungsbasis liegen in
  einem gemeinsamen Datensatz. Die bisherigen SharedPreferences bleiben als
  Rückfallkopie erhalten; fehlerhafte Altbestände werden nicht überschrieben.
- Cloud: ein Dokument pro Bike einschließlich Setups und Historie, dazu ein
  Dokument für die Feldbibliothek und eines für die Bike-Reihenfolge.
  Gleichzeitige Änderungen an verschiedenen Setups desselben Bikes werden
  vorsichtshalber als Konflikt behandelt. Eine spätere feinere Aufteilung ist möglich.
- Die Cloud prüft die erwartete Revision atomar. Löschungen bleiben als
  Löschmarkierungen erhalten. Die letzten 20 vorherigen Versionen je Dokument
  können wiederhergestellt werden. Die Oberfläche zeigt die 100 neuesten
  historischen Einträge des Kontos.
- Bilder: private, unveränderliche Objekte anhand ihres Inhalts-Hashs,
  maximal 5 MB pro Datei. Auf dem zweiten Gerät werden sie lokal eingebettet
  gespeichert, damit sie offline angezeigt werden können.
- Abmelden und Anmelden trennen lokale Kontobestände. Beim Gastimport entstehen
  neue IDs; lokale Sicherungen sind als Kopien wiederherstellbar. Noch nicht
  synchronisierte Daten können nur auf dem ursprünglichen Gerät vorhanden sein.
- Automatische Übertragung nach Änderungen (2 Sekunden), bei Rückkehr in die
  App und alle 45 Sekunden, solange die App läuft. Kein Betriebssystem-Dienst
  für Uploads bei geschlossener App. Im Browser dieselbe Installation nicht
  gleichzeitig in mehreren Tabs bearbeiten: Ein veralteter zweiter Tab darf
  lokale Änderungen nicht überschreiben und muss nach Export eventueller
  ungespeicherter Änderungen neu geladen werden.
- Ein verlorenes anonymes Anmeldetoken ist nicht durch die UID ersetzbar.
  Ohne verknüpfte E-Mail bleiben Daten nach Geräteverlust eventuell unerreichbar.
- Konto-Selbstlöschung, Google/Apple-Anmeldung und automatisierte externe
  Betreiber-Backups sind noch nicht Bestandteil dieser ersten Version.
  Vor öffentlicher Veröffentlichung Kontolöschung und Betreiber-Backups ergänzen.
- Der lokale Versionsbestand und verwaiste Bilder werden noch nicht automatisch
  bereinigt. Speicherverbrauch überwachen; vor einer Bereinigung Referenzen aus
  der Cloud-Historie berücksichtigen. Nicht pauschal anonyme Accounts löschen.

## Vor öffentlicher Veröffentlichung

- Mit zwei realen Nutzern prüfen, dass Daten- und Bildzugriffe des anderen
  Kontos abgewiesen werden. RPC-Konflikttest ebenfalls gegen Supabase ausführen.
- E-Mail-Versand, Ratenlimits und Schutz vor massenhaften anonymen Anmeldungen
  konfigurieren. Falls CAPTCHA aktiviert wird, muss dessen Token-Übergabe in
  der App ergänzt werden; diese Version hat noch keine CAPTCHA-Oberfläche.
- Separate Backups von Datenbank **und Bilddateien** einrichten und eine
  Wiederherstellung testen. Die Dokumenthistorie ist kein unabhängiges Backup.
- Kostenloses Supabase-Projekt: Inaktivitätspausen und Speicher-/Traffic-Limits
  beachten. Ein kostenloser Nutzeraccount muss nicht bedeuten, dass der
  App-Betrieb dauerhaft kostenlos bleibt.
- Native Zielplattformen einschließlich E-Mail-Codes und Dateifreigabe auf
  echten Geräten prüfen. Ein Web-Build beweist keine iOS-/Android-Funktion.

## Quellen

- https://supabase.com/docs/guides/auth/auth-anonymous
- https://supabase.com/docs/guides/auth/auth-smtp
- https://supabase.com/docs/guides/database/postgres/row-level-security
- https://supabase.com/pricing


# Google-Anmeldung aktivieren

Für diesen Einstieg brauchst du weder eine eigene Domain noch SMTP.
Die Anmeldung läuft über Google und die vorhandene Supabase-Projektadresse.
Google Client Secret nur in Supabase eintragen, niemals in die Flutter-App.

## 1. Google-Projekt und Zustimmung einrichten

1. https://console.cloud.google.com/ öffnen und ein Projekt anlegen oder auswählen,
   beispielsweise **Bike Setup Tracker**.
2. **Google Auth Platform** öffnen. Falls noch nicht eingerichtet, **Get started**
   wählen. App-Name, Support-E-Mail und Entwickler-Kontaktadresse hinterlegen.
3. Zielgruppe **External / Extern** auswählen. Für den Anfang im Testmodus bleiben
   und unter **Audience → Test users** die Google-Adressen deiner Tester hinzufügen.
4. Nur die Anmeldedaten `openid`, E-Mail und Profil verwenden. Keine Drive-,
   Kalender- oder sonstigen zusätzlichen Berechtigungen anfordern.

## 2. OAuth-Client erstellen

Unter **Clients → Create client**:

- Typ: **Web application**. Auch die native App verwendet in dieser Umsetzung
  den Browser und Supabase als Vermittler.
- Name: beispielsweise **Bike Tracker Supabase**.
- Unter **Authorized redirect URIs** exakt eintragen:

```text
https://dcrkfiooddkbzljibomo.supabase.co/auth/v1/callback
```

- Für lokale Web-Tests als **Authorized JavaScript origin** eintragen:

```text
http://localhost:5173
```

Client erstellen. **Client ID** und **Client secret** werden im nächsten Schritt
direkt in Supabase eingetragen. Das Secret nicht hier im Chat veröffentlichen.

## 3. Google in Supabase aktivieren

1. **Authentication → Sign In / Providers → Google** öffnen.
2. Google aktivieren, Client ID und Client Secret aus Schritt 2 eintragen, speichern.
3. Unter den Auth-Einstellungen **Allow manual linking / Manual identity linking**
   einschalten. Das ist für **Mit Google absichern** erforderlich, damit die
   anonyme Nutzer-ID und ihre Daten erhalten bleiben.
4. Anonyme Anmeldung eingeschaltet lassen.

## 4. Rückkehr zur App erlauben

In **Authentication → URL Configuration → Redirect URLs**:

```text
http://localhost:5173/
bikesetuptracker://auth/callback
http://127.0.0.1:43827/auth/callback**
```

- Erste Adresse: Web-Test. App mit `flutter run -d chrome --web-port 5173` starten.
  Wichtig: immer denselben Host und Port verwenden, damit Browser-Daten und der
  Sicherheitsnachweis der begonnenen Anmeldung wiedergefunden werden.
- Zweite Adresse: Android/iOS. Die Rückleitung ist im App-Projekt registriert.
- Dritte Adresse: Windows/macOS/Linux. Die App öffnet für maximal 15 Minuten
  einen lokalen Listener, ausschließlich auf `127.0.0.1`. Ein einmaliger
  Versuchscode in der Rückleitung wird zusätzlich zu PKCE geprüft. Bei belegtem
  Port meldet die Anmeldung einen Fehler; keine Firewall-Öffnung nach außen nötig.

Die **Site URL** für diesen Web-Test auf `http://localhost:5173/` setzen.
Bei einer späteren Veröffentlichung die tatsächliche Web-Adresse ergänzen
und als Site URL setzen. Die App sendet ihre konkrete Rückleitungsadresse mit;
eine fehlende Freigabe kann trotzdem wieder zur falschen Site URL führen.

Die Google-Redirect-URI aus Schritt 2 und die App-Rückleitungen aus Schritt 4
haben unterschiedliche Aufgaben und müssen in den jeweiligen Diensten stehen.

## 5. Prüfen

App wegen der nativen Rückleitung vollständig neu bauen/starten.

1. Mit vorhandenem Gastbestand **Konto & Datensicherung → Mit Google absichern**
   wählen. Im Browser Google-Konto auswählen und zur App zurückkehren.
2. In Supabase Authentication prüfen: dieselbe Nutzer-ID, jetzt mit Google.
   Die bestehenden Bikes und Bilder müssen erhalten bleiben.
3. In einem zweiten Browserprofil/einem zweiten Gerät **Mit Google anmelden**
   wählen. Mit demselben Google-Konto anmelden, Daten und Bilder prüfen.
4. Wenn auf diesem zweiten Gerät schon Gastdaten vorhanden waren, liegen diese
   unter **Lokale Sicherungen** zur ausdrücklichen Übernahme als Kopien bereit.
5. Abbruch, Browser-Zurück und Neuladen während der Anmeldung testen.
6. Wird bei der Verknüpfung ein bereits zu einem anderen Account gehörendes
   Google-Konto gewählt, Anmeldung abbrechen und **Mit Google anmelden** verwenden.
   Die App hängt die Gastdaten niemals ungeprüft an eine andere Identität um.

## Typische Meldungen

- `redirect_uri_mismatch`: Google muss die Supabase-Callback-Adresse aus Schritt 2 kennen.
- Rückkehr zu `localhost:3000`: Supabase-Rückleitungen und Site URL aus Schritt 4 prüfen.
- Provider deaktiviert: Google in Supabase aktivieren und Zugangsdaten speichern.
- Manual linking deaktiviert: Schritt 3.3 aktivieren.
- Zugriff verweigert im Testmodus: verwendete Google-Adresse als Testnutzer hinzufügen.
- Anmeldung abgelaufen: in der App abbrechen und erneut starten.

Die E-Mail-Code-Oberfläche wurde durch Google ersetzt. Vorhandene
E-Mail-Accounts können, solange sie angemeldet sind, Google verknüpfen.
E-Mail-API-Methoden bleiben zur Kompatibilität erhalten, sind aber kein
öffentlicher Registrierungsweg dieser Version.

Die Tests simulieren Supabase-Antworten. Echte Google-Anmeldung kann erst nach
Konfiguration der Client-Zugangsdaten geprüft werden. Android/iOS/macOS sind
hier nicht auf echten Geräten geprüft. Vor öffentlichem Betrieb Google-Zielgruppe
und gegebenenfalls erforderliche Branding-Prüfung abschließen; Testmodus ist
kein öffentlicher Release.

Quellen:
- https://supabase.com/docs/guides/auth/social-login/auth-google
- https://supabase.com/docs/guides/auth/auth-identity-linking
- https://supabase.com/docs/guides/auth/redirect-urls


# Kontolöschung aktivieren

Die App enthält jetzt „Konto und Daten löschen“, auch für Gastkonten. Vorher wird
ein Export angeboten; die endgültige Bestätigung erfordert die Eingabe `DELETE`.
Die Datenschutz-HTML wurde nicht verändert.

## Supabase einrichten (vor dem App-Deployment)

1. Im SQL Editor den Inhalt von
   `supabase/migrations/202609160001_account_deletion.sql` ausführen.
   Voraussetzung ist die vorhandene Accounts-and-Sync-Migration.
2. Edge Function **delete-account** bereitstellen. Mit Supabase CLI:

   ```sh
   supabase functions deploy delete-account --project-ref dcrkfiooddkbzljibomo --no-verify-jwt
   ```

   Alternativ im Dashboard eine Edge Function mit diesem Namen erstellen und
   **beide** Dateien `index.ts` und `handler.mjs` aus
   `supabase/functions/delete-account/` übernehmen. JWT-Prüfung im Gateway
   ausschalten: Die Funktion prüft JWT-Signatur, Ablauf und Issuer selbst über
   Supabase Auth und validiert bestehende Nutzer zusätzlich mit `getUser`.
   Ohne gültiges Nutzertoken ist keine Löschung möglich.
3. `SUPABASE_URL` und `SUPABASE_SERVICE_ROLE_KEY` sind Standard-Secrets der
   gehosteten Edge-Umgebung. Den Service-Key niemals in Flutter oder GitHub-Code eintragen.
4. Erst danach die App veröffentlichen.

## Ablauf und Grenzen

- Ziel-ID ausschließlich aus verifiziertem JWT; fremde IDs im Request werden ignoriert.
- Löschmarker sperrt neue Dokument- und Bildschreibzugriffe, auch von anderen Geräten.
- Bilder werden stapelweise über die Storage-API gelöscht, einschließlich Unterordnern.
- Anschließend löscht die Admin-API das Auth-Konto. Fremdschlüssel löschen Dokumente,
  alte Versionen und Löschmarker automatisch.
- Teilfehler behalten den Löschmarker. Ein erneuter Versuch setzt die Löschung fort.
  Abbruch kann bereits gelöschte Bilder nicht wiederherstellen.
- Erfolgreiche Cloud-Löschung wird lokal markiert. Abmeldung, lokale Kontosicherungen,
  alte Migrationskopien und zugeordnete app-eigene Bilddateien werden bereinigt.
  Andere gespeicherte Konten bleiben bestehen. Gemeinsam referenzierte Bilddateien
  werden für diese Konten erhalten. Externe Originale und Exporte werden nicht gelöscht.
- Danach bleibt die App auch nach Neustart pausiert. Erst „Neu als Gast starten“
  erzeugt wieder ein Konto. Während eines offenen Löschvorgangs sind Änderungen gesperrt.
- Lokale Kopien auf anderen Geräten sowie Anbieter-Backups und Logs werden nicht
  fern-gelöscht. Die jeweiligen Aufbewahrungsfristen gelten weiterhin.
- Bei abgelaufener/verlorener Sitzung während einer unvollständigen Löschung ist
  gegebenenfalls eine administrative Fortsetzung nötig. Niemals lokale Daten vor
  einer bestätigten Cloud-Löschung als erfolgreich gelöscht behandeln.

## Prüfung nach Einrichtung

Mit einem separaten Testkonto mehrere Bikes/Bilder und Versionen anlegen, Export
speichern, dann löschen. Auth-User, Dokumente, Historie und alle Bilder im UID-Ordner
müssen fehlen. App neu laden: keine automatische Gastanmeldung. Danach explizit
neu starten. Ebenfalls Fehler/Wiederholung und zwei parallel geöffnete Geräte testen.

Lokal werden die Flutter-Tests und die Handler-Tests mit
`node --test supabase/functions/delete-account/handler.test.mjs` ausgeführt.
Diese ersetzen keinen Integrationstest der Migration und Storage-Löschung in Supabase.
